# SentinelHome earthquake vulnerability benchmark

This notebook documents the earthquake training data, removes post-earthquake leakage, compares tree-based classifiers with stratified cross-validation, evaluates them on a held-out test set, and saves the best complete pipeline to `backend/ml/models/earthquake_damage_model.pkl`.

The model output is an earthquake `damage_grade` prediction and class probabilities. Live USGS hazard severity and household vulnerability can be combined by the application risk layer after this prediction.

## 1. Setup and project paths

The path resolution works from the repository root, `backend`, or `backend/ml/src`;

In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError(
        "CatBoost is required for this benchmark. Install it with `pip install catboost`."
    ) from exc

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_project_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for candidate in candidates:
        if (candidate / "backend" / "ml" / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not locate the SentinelHome repository root")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "backend" / "ml" / "data" / "raw" / "earthquake_final_training_dataset_200k.csv"
MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Model output directory:", MODEL_DIR)

Project root: D:\Vs code Projects\Sentinel-Home
Dataset: D:\Vs code Projects\Sentinel-Home\backend\ml\data\raw\earthquake_final_training_dataset_200k.csv
Model output directory: D:\Vs code Projects\Sentinel-Home\backend\ml\models


## 2. Dataset definition and inspection

The CSV contains earthquake-building observations. Structural columns describe the building before the earthquake; hazard columns describe intensity and distance; `damage_grade` is the supervised target. The file is synthetic/physics-augmented according to its `data_source` field, so its results should be reported as an experiment rather than as measured deployment accuracy. The inspection cell prints row/column counts, data types, missing values, cardinality, numeric descriptive statistics, example values, duplicate count, class balance, and source balance.

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.head())

dataset_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=False),
}).sort_index()
display(dataset_summary)
print("Descriptive statistics and categorical value examples:")
display(df.select_dtypes(include="number").describe().transpose())
value_examples = pd.DataFrame({
    "example_values": {
        column: df[column].dropna().head(10000).astype(str).drop_duplicates().head(10).tolist()
        for column in df.columns
    }
})
display(value_examples)
print("Duplicate rows:", int(df.duplicated().sum()))
print("Target distribution:")
display(df["damage_grade"].value_counts(dropna=False).sort_index().to_frame("count"))
print("Data source distribution:")
display(df["data_source"].value_counts(dropna=False).to_frame("count"))

Rows: 1,599,952
Columns: 37


,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,count_floors_post_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,height_ft_post_eq,...,has_superstructure_other,condition_post_eq,technical_solution_proposed,magnitude,epicentral_distance_km,hypocentral_distance_km,pga_cm_s2,mmi,damage_grade,data_source
0,303801001111,30,3005,300502,2,0,5,200,14,0,...,0,Damaged-Rubble Clear-New building built,Reconstruction,4.0,32.52,33.54,25.40,3.48,1,synthetic_physics_augmented
1,301606020601,30,3002,300203,3,0,57,315,20,0,...,0,Damaged-Rubble unclear,Reconstruction,4.0,40.45,41.27,20.80,3.16,1,synthetic_physics_augmented
2,304108010261,30,3013,301302,2,2,43,768,20,20,...,0,Damaged-Repaired and used,Minor repair,4.0,43.22,43.99,19.48,3.06,1,synthetic_physics_augmented
3,301608013491,30,3002,300205,3,0,20,375,18,0,...,0,Damaged-Rubble unclear,Reconstruction,4.0,42.59,43.38,19.76,3.08,1,synthetic_physics_augmented
4,303405002801,30,3002,300209,3,1,5,450,24,8,...,0,Damaged-Repaired and used,Reconstruction,4.0,41.21,42.02,20.42,3.14,1,synthetic_physics_augmented


,dtype,missing_values,unique_values
age_building,int64,0,146
building_id,int64,0,199994
condition_post_eq,str,0,8
count_floors_post_eq,int64,0,10
count_floors_pre_eq,int64,0,9
damage_grade,int64,0,5
data_source,str,0,2
district_id,int64,0,11
epicentral_distance_km,float64,0,13835
foundation_type,str,0,5


Descriptive statistics and categorical value examples:


,count,mean,std,min,25%,50%,75%,max
building_id,1599952.0,2.607516e+11,5.801852e+10,1.201010e+11,2.219090e+11,2.463010e+11,3.036060e+11,3.667090e+11
district_id,1599952.0,2.576792e+01,5.807575e+00,1.200000e+01,2.200000e+01,2.400000e+01,3.000000e+01,3.600000e+01
vdcmun_id,1599952.0,2.582691e+03,5.811719e+02,1.201000e+03,2.204000e+03,2.410000e+03,3.010000e+03,3.611000e+03
ward_id,1599952.0,2.582745e+05,5.811726e+04,1.201010e+05,2.204020e+05,2.410040e+05,3.010050e+05,3.611080e+05
count_floors_pre_eq,1599952.0,2.086663e+00,6.553495e-01,1.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,9.000000e+00
count_floors_post_eq,1599952.0,1.249192e+00,1.063938e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00,9.000000e+00
age_building,1599952.0,2.427529e+01,6.492399e+01,0.000000e+00,9.000000e+00,1.600000e+01,2.700000e+01,9.990000e+02
plinth_area_sq_ft,1599952.0,4.064970e+02,2.248234e+02,7.000000e+01,2.800000e+02,3.580000e+02,4.700000e+02,5.000000e+03
height_ft_pre_eq,1599952.0,1.604286e+01,5.496310e+00,6.000000e+00,1.200000e+01,1.600000e+01,1.800000e+01,9.900000e+01
height_ft_post_eq,1599952.0,9.847000e+00,8.580685e+00,0.000000e+00,0.000000e+00,1.100000e+01,1.600000e+01,9.900000e+01


,example_values
building_id,"[303801001111, 301606020601, 304108010261, 301..."
district_id,[30]
vdcmun_id,"[3005, 3002, 3013, 3003, 3004, 3009, 3006, 300..."
ward_id,"[300502, 300203, 301302, 300205, 300209, 30050..."
count_floors_pre_eq,"[2, 3, 1, 4, 5, 6, 7]"
count_floors_post_eq,"[0, 2, 1, 3, 5, 4, 6, 7]"
age_building,"[5, 57, 43, 20, 22, 75, 2, 3, 7, 10]"
plinth_area_sq_ft,"[200, 315, 768, 375, 450, 540, 300, 337, 130, ..."
height_ft_pre_eq,"[14, 20, 18, 24, 22, 12, 10, 16, 9, 21]"
height_ft_post_eq,"[0, 20, 8, 16, 12, 18, 10, 14, 24, 22]"


Duplicate rows: 0
Target distribution:


,count
damage_grade,
1,1126167
2,246240
3,106867
4,48043
5,72635


Data source distribution:


,count
data_source,
synthetic_physics_augmented,1399958
real_observed,199994


## 3. Project-aligned feature contract

Post-earthquake fields are excluded because they are unavailable when SentinelHome makes a pre-disaster assessment and would leak the answer. IDs and source labels are excluded because they are identifiers, not household risk features. The model predicts building `damage_grade` from pre-earthquake structural information plus live earthquake hazard values. Household size and vulnerable-member information belong in the downstream urgency/risk layer: they affect the household's response priority, but they do not describe physical building damage. Every candidate feature is inspected and automatically removed if constant in the training file.

In [3]:
TARGET = "damage_grade"
IDENTIFIER_COLUMNS = ["building_id", "district_id", "vdcmun_id", "ward_id", "data_source"]
LEAKAGE_COLUMNS = [
    "count_floors_post_eq", "height_ft_post_eq",
    "condition_post_eq", "technical_solution_proposed",
]
PRE_EARTHQUAKE_STRUCTURAL = [
    "count_floors_pre_eq", "age_building", "plinth_area_sq_ft",
    "height_ft_pre_eq", "land_surface_condition", "foundation_type",
    "roof_type", "ground_floor_type", "other_floor_type",
    "position", "plan_configuration",
]
SUPERSTRUCTURE_COLUMNS = [c for c in df.columns if c.startswith("has_superstructure_")]
LIVE_HAZARD_COLUMNS = [
    "magnitude", "epicentral_distance_km",
    "hypocentral_distance_km", "mmi",
]
CANDIDATE_FEATURES = [
    *PRE_EARTHQUAKE_STRUCTURAL,
    *SUPERSTRUCTURE_COLUMNS,
    *LIVE_HAZARD_COLUMNS,
]
CANDIDATE_FEATURES = [c for c in CANDIDATE_FEATURES if c in df.columns]
CONSTANT_FEATURES = [c for c in CANDIDATE_FEATURES if df[c].nunique(dropna=False) <= 1]
FEATURES = [c for c in CANDIDATE_FEATURES if c not in CONSTANT_FEATURES]

print("Excluded identifier columns:", IDENTIFIER_COLUMNS)
print("Excluded leakage columns:", LEAKAGE_COLUMNS)
print("Removed constant candidate features:", CONSTANT_FEATURES)
print("Final model features:", FEATURES)
print("Final feature count:", len(FEATURES))

df = df.dropna(subset=[TARGET]).copy()
X = df[FEATURES].copy()
y_raw = df[TARGET].astype(int).copy()
CLASS_LABELS = sorted(y_raw.unique().tolist())
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASS_LABELS)}
INDEX_TO_CLASS = {index: label for label, index in CLASS_TO_INDEX.items()}
y = y_raw.map(CLASS_TO_INDEX).astype(int)
print("Output classes (damage_grade):", CLASS_LABELS)

Excluded identifier columns: ['building_id', 'district_id', 'vdcmun_id', 'ward_id', 'data_source']
Excluded leakage columns: ['count_floors_post_eq', 'height_ft_post_eq', 'condition_post_eq', 'technical_solution_proposed']
Removed constant candidate features: []
Final model features: ['count_floors_pre_eq', 'age_building', 'plinth_area_sq_ft', 'height_ft_pre_eq', 'land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'has_superstructure_adobe_mud', 'has_superstructure_mud_mortar_stone', 'has_superstructure_stone_flag', 'has_superstructure_cement_mortar_stone', 'has_superstructure_mud_mortar_brick', 'has_superstructure_cement_mortar_brick', 'has_superstructure_timber', 'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered', 'has_superstructure_rc_engineered', 'has_superstructure_other', 'magnitude', 'epicentral_distance_km', 'hypocentral_distance_km', 'mmi']
Final feature count: 26
Output classes 

## 4. Stratified train/test data

The held-out test set is never used by GridSearchCV. Grid search uses stratified 2-fold cross-validation on the complete training split, while the final selected estimator is refit on that complete training split before test evaluation. Two folds preserve a genuine multiple-fold comparison while keeping the full-data experiment practical on a local machine.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_grid, y_grid = X_train, y_train

numeric_features = X_grid.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_grid.select_dtypes(exclude=["number"]).columns.tolist()
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Grid-search rows (full training split):", len(X_grid))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Train rows: 1279961 Test rows: 319991
Grid-search rows (full training split): 1279961
Numeric features: ['count_floors_pre_eq', 'age_building', 'plinth_area_sq_ft', 'height_ft_pre_eq', 'has_superstructure_adobe_mud', 'has_superstructure_mud_mortar_stone', 'has_superstructure_stone_flag', 'has_superstructure_cement_mortar_stone', 'has_superstructure_mud_mortar_brick', 'has_superstructure_cement_mortar_brick', 'has_superstructure_timber', 'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered', 'has_superstructure_rc_engineered', 'has_superstructure_other', 'magnitude', 'epicentral_distance_km', 'hypocentral_distance_km', 'mmi']
Categorical features: ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration']


## 5. GPU configuration

XGBoost uses `device='cuda'` when CUDA is available. CatBoost uses `task_type='GPU'` when CUDA is available. Decision Tree and Random Forest remain CPU models; their parallelism is enabled where supported. If no compatible GPU is available, the notebook falls back automatically to CPU instead of failing.

In [5]:
def cuda_available():
    if os.getenv("SENTINEL_USE_GPU", "auto").lower() == "false":
        return False
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        return False
    try:
        result = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True, timeout=10)
        return result.returncode == 0 and bool(result.stdout.strip())
    except Exception:
        return False

USE_GPU = cuda_available()
XGB_DEVICE = "cuda" if USE_GPU else "cpu"
CATBOOST_TASK_TYPE = "GPU" if USE_GPU else "CPU"
print("GPU enabled:", USE_GPU)
print("XGBoost device:", XGB_DEVICE)
print("CatBoost task type:", CATBOOST_TASK_TYPE)

GPU enabled: True
XGBoost device: cuda
CatBoost task type: GPU


## 6. Model comparison with GridSearchCV

Weighted F1 is the primary selection metric because the damage grades are imbalanced. Accuracy, precision, recall, F1, and the classification report are still recorded on the untouched test set.

In [6]:
def make_pipeline(model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

models_and_grids = {
    "Decision Tree": (
        make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
        {
            "model__criterion": ["gini"],
            "model__max_depth": [12, 24],
            "model__min_samples_leaf": [1],
        },
    ),
    "Random Forest": (
        make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None],
            "model__min_samples_leaf": [1],
        },
    ),
    "XGBoost": (
        make_pipeline(XGBClassifier(
            random_state=RANDOM_STATE, n_jobs=-1, device=XGB_DEVICE,
            tree_method="hist", eval_metric="mlogloss",
        )),
        {
            "model__n_estimators": [150, 300],
            "model__max_depth": [6],
            "model__learning_rate": [0.1],
            "model__subsample": [0.9],
        },
    ),
    "CatBoost": (
        make_pipeline(CatBoostClassifier(
            random_seed=RANDOM_STATE, verbose=False,
            task_type=CATBOOST_TASK_TYPE, thread_count=-1,
            loss_function="MultiClass",
        )),
        {
            "model__iterations": [200, 400],
            "model__depth": [6],
            "model__learning_rate": [0.1],
        },
    ),
}

searches = {}
cv_results = []
all_grid_results = []
for name, (pipeline, grid) in models_and_grids.items():
    print(f"\nRunning {name}...")
    search = GridSearchCV(
        estimator=pipeline, param_grid=grid, scoring="f1_weighted",
        cv=2, n_jobs=1 if USE_GPU else -1, verbose=2,
        refit=True, return_train_score=False, error_score="raise",
    )
    search.fit(X_grid, y_grid)
    searches[name] = search
    for params, mean_score, std_score, rank in zip(
        search.cv_results_["params"],
        search.cv_results_["mean_test_score"],
        search.cv_results_["std_test_score"],
        search.cv_results_["rank_test_score"],
    ):
        all_grid_results.append({
            "model": name,
            "mean_cv_weighted_f1": mean_score,
            "std_cv_weighted_f1": std_score,
            "rank": rank,
            "parameters": params,
        })
    cv_results.append({
        "model": name,
        "best_cv_weighted_f1": search.best_score_,
        "best_params": search.best_params_,
    })
    print("Best CV F1:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

cv_summary = pd.DataFrame(cv_results).sort_values("best_cv_weighted_f1", ascending=False)
grid_search_results = pd.DataFrame(all_grid_results).sort_values(
    ["mean_cv_weighted_f1", "rank"], ascending=[False, True]
).reset_index(drop=True)
display(cv_summary)
display(grid_search_results)


Running Decision Tree...
Fitting 2 folds for each of 2 candidates, totalling 4 fits
[CV] END model__criterion=gini, model__max_depth=12, model__min_samples_leaf=1; total time=   7.6s
[CV] END model__criterion=gini, model__max_depth=12, model__min_samples_leaf=1; total time=  10.5s
[CV] END model__criterion=gini, model__max_depth=24, model__min_samples_leaf=1; total time=  17.4s
[CV] END model__criterion=gini, model__max_depth=24, model__min_samples_leaf=1; total time=  15.7s
Best CV F1: 0.844
Best parameters: {'model__criterion': 'gini', 'model__max_depth': 12, 'model__min_samples_leaf': 1}

Running Random Forest...
Fitting 2 folds for each of 2 candidates, totalling 4 fits
[CV] END model__max_depth=None, model__min_samples_leaf=1, model__n_estimators=100; total time=  39.4s
[CV] END model__max_depth=None, model__min_samples_leaf=1, model__n_estimators=100; total time=  30.3s
[CV] END model__max_depth=None, model__min_samples_leaf=1, model__n_estimators=200; total time= 1.1min
[CV] EN

,model,best_cv_weighted_f1,best_params
2,XGBoost,0.852002,"{'model__learning_rate': 0.1, 'model__max_dept..."
3,CatBoost,0.850014,"{'model__depth': 6, 'model__iterations': 400, ..."
1,Random Forest,0.849346,"{'model__max_depth': None, 'model__min_samples..."
0,Decision Tree,0.843962,"{'model__criterion': 'gini', 'model__max_depth..."


,model,mean_cv_weighted_f1,std_cv_weighted_f1,rank,parameters
0,XGBoost,0.852002,0.000193,1,"{'model__learning_rate': 0.1, 'model__max_dept..."
1,CatBoost,0.850014,0.000061,1,"{'model__depth': 6, 'model__iterations': 400, ..."
2,Random Forest,0.849346,0.000087,1,"{'model__max_depth': None, 'model__min_samples..."
3,XGBoost,0.849109,0.000256,2,"{'model__learning_rate': 0.1, 'model__max_dept..."
4,Random Forest,0.848605,0.000151,2,"{'model__max_depth': None, 'model__min_samples..."
5,CatBoost,0.846541,0.000124,2,"{'model__depth': 6, 'model__iterations': 200, ..."
6,Decision Tree,0.843962,0.000308,1,"{'model__criterion': 'gini', 'model__max_depth..."
7,Decision Tree,0.830125,0.000983,2,"{'model__criterion': 'gini', 'model__max_depth..."


## 7. Held-out evaluation and final model selection

In [7]:
test_rows = []
for name, search in searches.items():
    predictions = search.best_estimator_.predict(X_test)
    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, predictions, average="weighted", zero_division=0),
        "best_cv_weighted_f1": search.best_score_,
    })

comparison = pd.DataFrame(test_rows).sort_values("f1_weighted", ascending=False).reset_index(drop=True)
display(comparison)
best_name = comparison.iloc[0]["model"]
best_search = searches[best_name]
best_estimator = clone(models_and_grids[best_name][0]).set_params(**best_search.best_params_)
print(f"Refitting {best_name} on all {len(X_train):,} training rows...")
best_estimator.fit(X_train, y_train)
best_predictions = best_estimator.predict(X_test)
print("Selected model:", best_name)
print("\nClassification report:")
print(classification_report(y_test, best_predictions, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, best_predictions))

,model,accuracy,precision_weighted,recall_weighted,f1_weighted,best_cv_weighted_f1
0,XGBoost,0.852596,0.856110,0.852596,0.852990,0.852002
1,Random Forest,0.852005,0.853239,0.852005,0.852107,0.849346
2,CatBoost,0.850565,0.853943,0.850565,0.850718,0.850014
3,Decision Tree,0.844577,0.848069,0.844577,0.845120,0.843962


Refitting XGBoost on all 1,279,961 training rows...
Selected model: XGBoost

Classification report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95    225234
           1       0.68      0.70      0.69     49248
           2       0.57      0.50      0.53     21373
           3       0.39      0.32      0.35      9609
           4       0.56      0.81      0.66     14527

    accuracy                           0.85    319991
   macro avg       0.63      0.66      0.64    319991
weighted avg       0.86      0.85      0.85    319991

Confusion matrix:
[[212776  10754   1142    234    328]
 [  7336  34431   5595    790   1096]
 [   919   4944  10713   2000   2797]
 [   247    150    926   3105   5181]
 [   220     73    542   1894  11798]]


## 8. Save the best model and its contract

The `.pkl` contains the fitted preprocessing pipeline and classifier together. The backend must pass a DataFrame with the saved `input_features` and must interpret the returned class labels as `damage_grade`.

In [8]:
model_path = MODEL_DIR / "earthquake_damage_model.pkl"
comparison_path = MODEL_DIR / "earthquake_model_comparison.csv"
grid_results_path = MODEL_DIR / "earthquake_grid_search_results.csv"
metadata_path = MODEL_DIR / "earthquake_model_metadata.json"

joblib.dump(best_estimator, model_path)
comparison.to_csv(comparison_path, index=False)
grid_search_results.to_csv(grid_results_path, index=False)
metadata = {
    "model_role": "pre-disaster building damage classification",
    "model_name": best_name,
    "target": TARGET,
    "class_labels": CLASS_LABELS,
    "class_to_index": CLASS_TO_INDEX,
    "index_to_class": INDEX_TO_CLASS,
    "input_features": FEATURES,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "excluded_identifier_columns": IDENTIFIER_COLUMNS,
    "excluded_leakage_columns": LEAKAGE_COLUMNS,
    "constant_features_removed": CONSTANT_FEATURES,
    "dataset_path": str(DATA_PATH),
    "selection_metric": "weighted_f1",
    "gpu_used": USE_GPU,
    "best_parameters": best_search.best_params_,
    "downstream_risk_layer_inputs": [
        "household_size", "vulnerable_members",
        "location", "latitude", "longitude"
    ],
}
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print("Saved model:", model_path)
print("Saved comparison:", comparison_path)
print("Saved grid-search results:", grid_results_path)
print("Saved metadata:", metadata_path)

Saved model: D:\Vs code Projects\Sentinel-Home\backend\ml\models\earthquake_damage_model.pkl
Saved comparison: D:\Vs code Projects\Sentinel-Home\backend\ml\models\earthquake_model_comparison.csv
Saved grid-search results: D:\Vs code Projects\Sentinel-Home\backend\ml\models\earthquake_grid_search_results.csv
Saved metadata: D:\Vs code Projects\Sentinel-Home\backend\ml\models\earthquake_model_metadata.json


## 9. Backend inference example

Use the saved pipeline rather than rebuilding preprocessing in Flask. The application should build one row using the exact `input_features` from `earthquake_model_metadata.json`, call the helper below, and then pass the vulnerability result to the composite risk layer.

In [9]:
def predict_damage(model_path, metadata_path, household_row):
    """Predict damage grade for one already-normalized household row."""
    model = joblib.load(model_path)
    metadata = json.loads(Path(metadata_path).read_text(encoding="utf-8"))
    expected_features = metadata["input_features"]
    missing_features = [
        feature for feature in expected_features
        if feature not in household_row.columns
    ]
    if missing_features:
        raise ValueError(f"Missing model features: {missing_features}")

    model_input = household_row.loc[:, expected_features]
    predicted_index = int(model.predict(model_input)[0])
    damage_grade = metadata["index_to_class"][str(predicted_index)]
    probabilities = model.predict_proba(model_input)[0].tolist()
    return {
        "damage_grade": int(damage_grade),
        "probabilities": probabilities,
    }

# Example from Flask after collecting all saved input_features:
# result = predict_damage(
#     MODEL_DIR / "earthquake_damage_model.pkl",
#     MODEL_DIR / "earthquake_model_metadata.json",
#     household_row,
# )